In [ ]:
import subprocess, os, sys

print("Installing SUMO...")
r = subprocess.run(['apt-get','install','-y','sumo','sumo-tools'],
                   capture_output=True, text=True)
print("SUMO installed" if r.returncode == 0 else f"SUMO failed: {r.stderr}")

os.environ['SUMO_HOME'] = '/usr/share/sumo'

print("Unzipping codebase...")
subprocess.run(['unzip','-q','-o',
    '/kaggle/input/smartmarl-codebase/smartmarl_kaggle.zip',
    '-d','/kaggle/working/'], capture_output=True)

os.chdir('/kaggle/working')
os.makedirs('results/raw', exist_ok=True)
os.makedirs('checkpoints', exist_ok=True)

print("Verifying SUMO mode...")
r = subprocess.run(['python','train.py','--scenario','standard',
    '--seed','99','--episodes','1'],
    capture_output=True, text=True, cwd='/kaggle/working')
if 'Mock mode: False' in r.stdout:
    print("CONFIRMED: Real SUMO mode")
else:
    print("ERROR: Still in mock mode")
    print(r.stdout[-500:])
    print(r.stderr[-500:])
    sys.exit(1)

In [ ]:
import subprocess, os, glob, shutil

SEEDS = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
SCENARIO = 'standard'
ABLATION = 'full'

for seed in SEEDS:
    result_path = f'/kaggle/working/results/raw/full_standard_seed{seed}.json'
    if os.path.exists(result_path):
        print(f"Seed {seed} already done, skipping")
        continue
    print(f"\n{'='*50}")
    print(f"Training: scenario={SCENARIO} ablation={ABLATION} seed={seed}")
    print(f"{'='*50}")
    r = subprocess.run(
        ['python','train.py',
         '--scenario', SCENARIO,
         '--ablation', ABLATION,
         '--seed', str(seed),
         '--episodes', '3000',
         '--result_json', result_path,
         '--skip_existing'],
        cwd='/kaggle/working',
        capture_output=True, text=True
    )
    print(r.stdout[-2000:])
    if r.returncode != 0:
        print(f"FAILED seed {seed}:", r.stderr[-500:])
    else:
        print(f"Seed {seed} COMPLETE")

print("\nAll seeds done for this notebook")

In [ ]:
import glob, shutil, os

os.makedirs('/kaggle/working/output', exist_ok=True)
saved = []
for f in glob.glob('/kaggle/working/results/raw/*.json'):
    dest = '/kaggle/working/output/' + os.path.basename(f)
    shutil.copy(f, dest)
    saved.append(os.path.basename(f))

print(f"Saved {len(saved)} result files to output:")
for name in sorted(saved):
    print(f"  {name}")